# CSC 2301 · Week 6 — Errors, debugging and testing

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/csc2301/notebooks/CSC2301_W6_errors_testing_production.ipynb)

**IUGB · BSc Mining / Petroleum Engineering** · Instructor: Laurent Kouadio

Every programmer writes bugs; good engineers **find them fast** and **stop them coming back**.
This week you read error messages, handle bad data without crashing, fix a broken function, and
test your code with `assert`. The data are 120 days of oil and water production from well KW-1
(synthetic), recorded by hand: some days are missing, misspelt or impossible.

**What you will practise:** reading a traceback · the common exceptions · `try` / `except` / `else` ·
`raise` · `assert` · a simple debugging method (print, isolate, fix, test).

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/csc2301/'
FILES = [('data/production_daily_kw1.csv', 'production_daily_kw1.csv'), ('tests/w6/q1.py', 'tests/q1.py'), ('tests/w6/q2.py', 'tests/q2.py'), ('tests/w6/q3.py', 'tests/q3.py'), ('tests/w6/q4.py', 'tests/q4.py'), ('tests/w6/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

## 1. Read the error from the bottom

A **traceback** lists the calls that led to the error. The **last line** says what went wrong;
the line above it says where. Run the cell and read it.

In [ ]:
def daily_total(oil, water):
    return oil + water

import traceback
try:
    daily_total("812.4", 95.0)    # one value is text: TypeError
except TypeError:
    traceback.print_exc()         # shows the traceback without stopping "Run all"

| exception | typical cause |
|---|---|
| `SyntaxError` | a typo: missing `:` or bracket |
| `NameError` | a name not defined yet (cell not run?) |
| `TypeError` | wrong type: `"3" + 1` |
| `ValueError` | right type, wrong value: `float("N/A")` |
| `ZeroDivisionError` | dividing by 0 |
| `KeyError` / `IndexError` | missing key / position past the end |

## 2. Handling an error: `try` / `except`

In [ ]:
for s in ["812.4", "N/A", "95,0"]:
    try:
        value = float(s)
    except ValueError:
        print(repr(s), "is not a number")
    else:
        print(repr(s), "->", value)

Catch **only the error you expect** (`except ValueError`), never a bare `except:`: it would also hide
your own bugs.

## 3. Raising an error yourself

When a function receives impossible input, **stop loudly** with `raise` rather than return a wrong number.

In [ ]:
def gor(gas_scf, oil_bbl):
    """Gas-oil ratio in scf/bbl."""
    if oil_bbl <= 0:
        raise ValueError(f"oil must be positive, got {oil_bbl}")
    return gas_scf / oil_bbl

print(gor(450_000, 900))
try:
    gor(450_000, 0)
except ValueError as e:
    print("Refused:", e)

## 4. Testing with `assert`

`assert condition` does nothing when the condition is true, and raises `AssertionError` when it is false.
A **test** is a few asserts on cases where you know the answer by hand.

In [ ]:
def test_gor():
    assert gor(1000, 10) == 100
    assert gor(0, 5) == 0
    try:
        gor(1000, 0)
        assert False, "gor should refuse oil = 0"
    except ValueError:
        pass
    print("all gor tests passed")

test_gor()

## 5. The data

In [ ]:
import csv

with open("production_daily_kw1.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "days")
for r in rows[:12]:
    print(r)

---
## Exercises

### Q1. A conversion that never crashes

Write `safe_float(s)` that returns `float(s)` when possible and `None` when `s` is not a valid number
(`'N/A'`, `''`, `'1,5'`…). Use `try` / `except ValueError`.

In [ ]:
def safe_float(s):
    ...

[safe_float(s) for s in ("12.5", "N/A", "", "1,5")]

In [ ]:
grader.check("q1")

### Q2. Water cut, with honest errors

The **water cut** is the fraction of water in the produced liquid: water / (oil + water).
Write `water_cut(oil, water)` that returns it, and **raises `ValueError`** if a volume is negative or if
oil + water is 0 (the well produced nothing: the water cut is undefined).

In [ ]:
def water_cut(oil, water):
    ...

water_cut(90.0, 10.0)

In [ ]:
grader.check("q2")

### Q3. How bad is the file?

Store in `n_days` the number of days in the file and in `n_bad` the number of **bad days**: oil or water
cannot be read (`safe_float` gives `None`) **or** is negative.

In [ ]:
n_days = 0
n_bad = 0
...
n_days, n_bad

In [ ]:
grader.check("q3")

### Q4. Debug this function

A colleague wrote `average_positive(values)`, which should return the mean of the values **greater
than 0**, or `None` when there are none. It has **several bugs**. Run it, compare with what you expect by hand,
and fix it. Add `print` lines inside the loop if it helps, then remove them.

In [ ]:
def average_positive(values):
    for v in values:
        total = 0
        count = 0
        if v >= 0:
            total += v
            count += 1
    return total / len(values)

average_positive([2.0, -1.0, 4.0, 0.0])    # should be 3.0

In [ ]:
grader.check("q4")

### Q5. Results from the clean days

Using only the **valid** days (oil and water readable and not negative):

* `cum_oil_bbl`: the cumulative oil production, in barrels;
* `worst_day`: the date (as text, e.g. `'2024-03-14'`) with the highest water cut, among valid days with
  oil + water > 0. Your `water_cut` can be used here; the shut-in days would make it raise an error.

In [ ]:
cum_oil_bbl = 0.0
worst_day = None
...
cum_oil_bbl, worst_day

In [ ]:
grader.check("q5")

### Going further (not checked)

* Write `test_water_cut()` with at least four asserts, including the two error cases, and run it.
* Why is it better that `water_cut(0, 0)` raises an error than returns 0?

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()